# Train `phi` — PAN924 dental report VLM (Colab · A100)

**Model:** `microsoft/Phi-3.5-vision-instruct`  ·  **Framework:** ms-swift (LoRA, bf16)

All 5 notebooks share the same settings, so the models are comparable. Checkpoints are saved on **Google Drive**, so if Colab disconnects you just **re-run the train cell and it continues** from the last checkpoint.
> **Note:** If you skip flash-attn, set ATTN_IMPL = 'eager' in the config cell.


## 1. Check the GPU
`Runtime -> Change runtime type -> A100 GPU`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install ms-swift and the per-model dependencies

In [ ]:
%pip install -q -U ms-swift accelerate
%pip install -q -U qwen_vl_utils timm einops av sentencepiece
# flash-attn makes the A100 much faster but takes a few minutes to build:
%pip install -q flash-attn --no-build-isolation
import swift, torch
print('ms-swift', swift.__version__, '| cuda available:', torch.cuda.is_available())


## 3. Mount Google Drive (this is what makes training resumable)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Put the code and dataset on the machine
Zip `vlm_report_dataset/` (with the 4620 images), upload it to Drive, and set the path below.

In [ ]:
import os
REPO_DIR = '/content/pan924'                              # will contain vlm_report_dataset/
DATA_ZIP = '/content/drive/MyDrive/pan924_dataset.zip'    # your uploaded zip

os.makedirs(REPO_DIR, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, 'vlm_report_dataset')):
    assert os.path.exists(DATA_ZIP), 'Upload your dataset zip to ' + DATA_ZIP
    !unzip -q {DATA_ZIP} -d {REPO_DIR}
os.chdir(REPO_DIR)   # the image paths in the data are relative to here

import json
first_line = open('vlm_report_dataset/converted/qwen/train.jsonl', encoding='utf-8').readline()
sample_image = json.loads(first_line)['images'][0]
assert os.path.exists(sample_image), 'sample image not found: ' + sample_image
print('OK - data and images found. Working directory:', os.getcwd())


## 5. Settings
Effective batch = `PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS` = 2 x 8 = 16 (same as the 3090 run).

In [ ]:
# ===== This model (the only part that differs between the 5 notebooks) =====
MODEL_NAME = "microsoft/Phi-3.5-vision-instruct"
MODEL_KEY = "phi"
USE_MAX_PIXELS = True
ATTN_IMPL = 'eager'      # 'eager' if flash-attn is not installed

# ===== Shared A100 settings (identical in all 5 notebooks = fair comparison) =====
TRAIN_TYPE = "lora"          # bf16 LoRA (A100 has the VRAM, no 4-bit needed)
TORCH_DTYPE = "bfloat16"
LORA_RANK = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = "all-linear"
FREEZE_VIT = "true"
NUM_EPOCHS = 2
LEARNING_RATE = "1e-4"
WEIGHT_DECAY = "0.1"
WARMUP_RATIO = "0.05"
LR_SCHEDULER = "cosine"
PER_DEVICE_BATCH_SIZE = 2    # A100. effective batch = 2 * 8 = 16 (same as the 3090 run)
GRAD_ACCUM_STEPS = 8
MAX_LENGTH = 4096
GRAD_CHECKPOINTING = "true"
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 12
SEED = 924
MAX_PIXELS = 1003520         # 1280*28*28 (higher than the 3090; lower first if OOM)

import os
OUTPUT_DIR = "/content/drive/MyDrive/pan924_runs/" + MODEL_KEY
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", OUTPUT_DIR)


## 6. Train (auto-resume)
Re-run this cell after a disconnect and it continues from the last checkpoint on Drive.

In [ ]:
import os
import glob
import subprocess

def find_last_checkpoint(folder):
    """Return the newest checkpoint-N folder, or None if there is none yet."""
    last_path = None
    last_step = -1
    for path in glob.glob(os.path.join(folder, "checkpoint-*")):
        number_text = os.path.basename(path).replace("checkpoint-", "")
        if number_text.isdigit() and int(number_text) > last_step:
            last_step = int(number_text)
            last_path = path
    return last_path

env = os.environ.copy()
if USE_MAX_PIXELS:
    env["MAX_PIXELS"] = str(MAX_PIXELS)

command = [
    "swift", "sft",
    "--model", MODEL_NAME,
    "--dataset", "vlm_report_dataset/converted/qwen/train.jsonl",
    "--val_dataset", "vlm_report_dataset/converted/qwen/val.jsonl",
    "--split_dataset_ratio", "0",
    "--train_type", TRAIN_TYPE,
    "--torch_dtype", TORCH_DTYPE,
    "--lora_rank", str(LORA_RANK),
    "--lora_alpha", str(LORA_ALPHA),
    "--lora_dropout", str(LORA_DROPOUT),
    "--target_modules", TARGET_MODULES,
    "--freeze_vit", FREEZE_VIT,
    "--num_train_epochs", str(NUM_EPOCHS),
    "--learning_rate", LEARNING_RATE,
    "--weight_decay", WEIGHT_DECAY,
    "--warmup_ratio", WARMUP_RATIO,
    "--lr_scheduler_type", LR_SCHEDULER,
    "--per_device_train_batch_size", str(PER_DEVICE_BATCH_SIZE),
    "--per_device_eval_batch_size", "1",
    "--gradient_accumulation_steps", str(GRAD_ACCUM_STEPS),
    "--max_length", str(MAX_LENGTH),
    "--gradient_checkpointing", GRAD_CHECKPOINTING,
    "--eval_strategy", "steps",
    "--eval_steps", str(EVAL_STEPS),
    "--save_strategy", "steps",
    "--save_steps", str(SAVE_STEPS),
    "--save_total_limit", str(SAVE_TOTAL_LIMIT),
    "--logging_steps", "5",
    "--seed", str(SEED),
    "--add_version", "false",
    "--output_dir", OUTPUT_DIR,
]

if ATTN_IMPL is not None:
    command += ["--attn_impl", ATTN_IMPL]

# Auto-resume: continue from the last checkpoint on Drive instead of starting over.
last_checkpoint = find_last_checkpoint(OUTPUT_DIR)
if last_checkpoint is not None:
    print("Continuing from checkpoint:", last_checkpoint)
    command += ["--resume_from_checkpoint", last_checkpoint]
else:
    print("Starting from the beginning.")

print(" ".join(command))
subprocess.run(command, env=env, check=True)
# If you get CUDA OOM: lower MAX_PIXELS (1003520 -> 802816 -> 602112), then
# PER_DEVICE_BATCH_SIZE 2 -> 1, and re-run this cell (it resumes).


## 7. Pick the best checkpoint by macro-F1 (not by loss)
Loss is dominated by the common `H` class, so we score every checkpoint on the validation set and keep the one with the best per-condition macro-F1.

In [ ]:
import os
import glob
import json
import subprocess

VAL_DATA = 'vlm_report_dataset/converted/qwen/val.jsonl'
EVAL_SCRIPT = 'vlm_report_dataset/scripts/eval_report.py'
ADAPTER_SCRIPT = 'vlm_report_dataset/training/tools/swift_pred_to_eval.py'

def list_checkpoints(folder):
    pairs = []
    for path in glob.glob(os.path.join(folder, 'checkpoint-*')):
        number_text = os.path.basename(path).replace('checkpoint-', '')
        if number_text.isdigit():
            pairs.append((int(number_text), path))
    pairs.sort()
    return [path for step, path in pairs]

env = os.environ.copy()
if USE_MAX_PIXELS:
    env['MAX_PIXELS'] = str(MAX_PIXELS)

scores = {}
for checkpoint in list_checkpoints(OUTPUT_DIR):
    infer_out = os.path.join(checkpoint, 'infer_val.jsonl')
    pred_out = os.path.join(checkpoint, 'preds_val.jsonl')
    metrics_out = os.path.join(checkpoint, 'metrics_val.json')
    subprocess.run(['swift', 'infer', '--adapters', checkpoint, '--val_dataset', VAL_DATA,
                    '--max_new_tokens', '1024', '--temperature', '0',
                    '--result_path', infer_out], env=env, check=True)
    subprocess.run(['python', ADAPTER_SCRIPT, '--val', VAL_DATA,
                    '--swift-result', infer_out, '--out', pred_out], check=True)
    subprocess.run(['python', EVAL_SCRIPT, '--gold', VAL_DATA, '--pred', pred_out,
                    '--out-json', metrics_out, '--tag', MODEL_KEY + '/val'], check=True)
    macro_f1 = json.load(open(metrics_out, encoding='utf-8'))['macro_f1']
    scores[checkpoint] = macro_f1
    print(checkpoint, 'macro-F1 =', round(macro_f1, 4))

BEST_CHECKPOINT = max(scores, key=scores.get)
print('\nBest checkpoint:', BEST_CHECKPOINT, '-> macro-F1', round(scores[BEST_CHECKPOINT], 4))


## 8. Final metrics on the test set
Reports accuracy / precision / recall / F1 (per condition and overall), FDI detection, hallucination and miss rates, exact-report match, and ROUGE-L on the text.

In [ ]:
import os
import json
import subprocess
import pandas as pd

TEST_DATA = 'vlm_report_dataset/converted/qwen/test.jsonl'
EVAL_SCRIPT = 'vlm_report_dataset/scripts/eval_report.py'
ADAPTER_SCRIPT = 'vlm_report_dataset/training/tools/swift_pred_to_eval.py'

env = os.environ.copy()
if USE_MAX_PIXELS:
    env['MAX_PIXELS'] = str(MAX_PIXELS)

infer_out = os.path.join(BEST_CHECKPOINT, 'infer_test.jsonl')
pred_out = os.path.join(BEST_CHECKPOINT, 'preds_test.jsonl')
metrics_out = os.path.join(OUTPUT_DIR, 'metrics_' + MODEL_KEY + '_test.json')

subprocess.run(['swift', 'infer', '--adapters', BEST_CHECKPOINT, '--val_dataset', TEST_DATA,
                '--max_new_tokens', '1024', '--temperature', '0',
                '--result_path', infer_out], env=env, check=True)
subprocess.run(['python', ADAPTER_SCRIPT, '--val', TEST_DATA,
                '--swift-result', infer_out, '--out', pred_out], check=True)
subprocess.run(['python', EVAL_SCRIPT, '--gold', TEST_DATA, '--pred', pred_out,
                '--out-json', metrics_out, '--tag', MODEL_KEY + '/test'], check=True)

metrics = json.load(open(metrics_out, encoding='utf-8'))
for name, value in metrics.items():
    if isinstance(value, (int, float)):
        print(name, '=', round(value, 4))

# per-condition table (does the model handle the rare conditions?)
pd.DataFrame(metrics['per_condition']).T.sort_values('support', ascending=False)


## 9. Compare all models
After all 5 notebooks finish, every `metrics_<key>_test.json` is on Drive:
```bash
python vlm_report_dataset/training/tools/compare_models.py /content/drive/MyDrive/pan924_runs/*/metrics_*_test.json
```